In [2]:
# Cell 1: Imports and Setup
import mne
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import colorama
import gc
from tqdm.auto import tqdm

# Add srcs to path to import misc
sys.path.append(os.path.abspath('../srcs'))
from misc import load_eeg_data, extract_and_map_events, EXCLUDED_SUBJECTS

BASE_DATA_PATH = "mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0"


# Ensure matplotlib plots display inline in the notebook
%matplotlib inline

# Set MNE logging level to 'WARNING' to reduce text output clutter
mne.set_log_level('WARNING')

print(f"MNE version: {mne.__version__}")
print(f"Excluded subjects: {EXCLUDED_SUBJECTS}")


MNE version: 1.12.1
Excluded subjects: [88, 89, 92, 100, 104, 106]


# Phase 2: Data Preprocessing, Validation, and Feature Extraction

## Step 1: Centralized metadata parsing for data validation

* This cell implements the optimized extraction logic. By setting `preload=False`, we avoid loading the entire dataset into memory, which is crucial for handling large EEG datasets efficiently. The event extraction is performed using MNE's built-in functions, and we map the annotations to a custom event ID dictionary for consistency across subjects and runs. **This approach ensures that we can quickly audit and analyze the event structure without unnecessary memory overhead.**

In [3]:
# Constants
SUBJECTS = range(1, 110)
RUNS = range(1, 15)
EVENT_ID = {"T0": 0, "T1": 1, "T2": 2}

metadata_records = []

print(f"Starting lightweight metadata extraction for {len(SUBJECTS)} subjects...")

for sub_id in tqdm(SUBJECTS, desc="Subjects"):
    sub_str = f"S{sub_id:03d}"
    sub_dir = os.path.join(BASE_DATA_PATH, sub_str)
    
    if not os.path.exists(sub_dir):
        continue
        
    for run_id in RUNS:
        run_str = f"R{run_id:02d}"
        file_name = f"{sub_str}{run_str}.edf"
        file_path = os.path.join(sub_dir, file_name)
        
        if not os.path.exists(file_path):
            continue
            
        try:
            # lightweight parse: preload=False only reads headers/annotations
            raw_header = mne.io.read_raw_edf(file_path, preload=False, verbose="ERROR")
            
            # Extract basic metadata
            sfreq = raw_header.info["sfreq"]
            n_channels = raw_header.info["nchan"]
            
            # Extract events
            events, _ = mne.events_from_annotations(raw_header, event_id=EVENT_ID, verbose="ERROR")
            
            metadata_records.append({
                "subject": sub_id,
                "run": run_id,
                "sfreq": sfreq,
                "n_channels": n_channels,
                "events": events,
                "n_events": len(events)
            })
            
        except Exception as e:
            print(f"Error parsing {sub_str}{run_str}: {e}")

# Consolidate into a single source of truth
df_metadata = pd.DataFrame(metadata_records)
print(f"Extraction complete. Metadata captured for {len(df_metadata)} files.")
df_metadata.head()

Starting lightweight metadata extraction for 109 subjects...


Subjects:   0%|          | 0/109 [00:00<?, ?it/s]

Extraction complete. Metadata captured for 1251 files.


,subject,run,sfreq,n_channels,events,n_events
0,1,1,160.0,64,"[[0, 0, 0]]",1
1,1,2,160.0,64,"[[0, 0, 0]]",1
2,1,3,160.0,64,"[[0, 0, 0], [672, 0, 2], [1328, 0, 0], [2000, ...",30
3,1,4,160.0,64,"[[0, 0, 0], [672, 0, 2], [1328, 0, 0], [2000, ...",30
4,1,5,160.0,64,"[[0, 0, 0], [672, 0, 2], [1328, 0, 0], [2000, ...",30


## Step 2: Channel Count & Sampling Rate Validation

This step consumes the centralized `df_metadata` to identify hardware or acquisition anomalies. We enforce two strict constraints:
1. **Sampling Frequency ($f_s$):** Must be exactly **160 Hz**.
2. **Channel Count:** Must be exactly **64 channels**.

Any deviation (like Subject 88's 128 Hz sampling) will cause tensor shape mismatches and spectral distortion in the downstream pipeline.

In [4]:
# Identify subjects with non-compliant sampling rates or channel counts
df_hardware_anomalies = df_metadata[
    (df_metadata["sfreq"] != 160.0) | (df_metadata["n_channels"] != 64)
].copy()

# Group by subject to list specific reasons for exclusion
hardware_exclusion_summary = (
    df_hardware_anomalies.groupby("subject")
    .agg(
        {
            "sfreq": lambda x: (
                f"Mismatched ({x.unique()[0]} Hz)"
                if x.unique()[0] != 160.0
                else "Correct"
            ),
            "n_channels": lambda x: (
                f"Mismatched ({x.unique()[0]} Ch)" if x.unique()[0] != 64 else "Correct"
            ),
        }
    )
    .reset_index()
)

# Generate the initial exclusion list from hardware failures
hardware_excluded_subjects = sorted(
    hardware_exclusion_summary["subject"].unique().tolist()
)

print(
    f"Detected {len(hardware_excluded_subjects)} subjects with hardware/acquisition anomalies."
)
if not hardware_exclusion_summary.empty:
    display(hardware_exclusion_summary)
else:
    print("All subjects pass hardware validation.")

Detected 3 subjects with hardware/acquisition anomalies.


,subject,sfreq,n_channels
0,88,Mismatched (128.0 Hz),Correct
1,92,Mismatched (128.0 Hz),Correct
2,100,Mismatched (128.0 Hz),Correct


## Step 3: Event Annotation & Marker Corruption Analysis

In this step, we analyze the trial structure of each run using the `events` extracted in Step 1. We enforce the following dataset standards:
- **Baseline Runs (1, 2):** Exactly **1 event** (continuous recording).
- **Task Runs (3-14):** Exactly **30 events** (15 T0 rest triggers, 15 task triggers).

Additionally, we incorporate known **literature-documented anomalies** for subjects like **S038** and **S089**, where event markers are present but physiologically unreliable or desynchronized.

In [5]:
# Define rules for event counts based on run type and literature
def check_event_integrity(row):
    sub = row["subject"]
    run = row["run"]
    n_ev = row["n_events"]

    anomalies = []

    # Standard Trial Count Checks
    if run in [1, 2]:
        if n_ev != 1:
            anomalies.append(f"R{run:02d}: Baseline count mismatch ({n_ev})")
    else:
        if n_ev != 30:
            anomalies.append(f"R{run:02d}: Task count mismatch ({n_ev}/30)")

    # Known Literature Anomalies (timing drift or labeling errors)
    literature_anomalies = {
        38: "Known annotation/timing drift",
        89: "Inconsistent labeling",
        92: "Labeling errors",
        100: "Inconsistent annotations",
        104: "Inconsistent annotations",
        106: "Inconsistent annotations",
    }

    if sub in literature_anomalies:
        # We only flag the subject once to keep the summary clean
        if run == 3:
            anomalies.append(f"Literature Flag: {literature_anomalies[sub]}")

    return "; ".join(anomalies) if anomalies else None


# Apply the integrity check to the centralized metadata
df_metadata["event_anomalies"] = df_metadata.apply(check_event_integrity, axis=1)

# Summarize subjects with any event issues
df_event_anomalies = df_metadata[df_metadata["event_anomalies"].notna()].copy()
event_exclusion_summary = (
    df_event_anomalies.groupby("subject")["event_anomalies"]
    .apply(lambda x: " | ".join(sorted(list(set(filter(None, x))))))
    .reset_index()
)

event_excluded_subjects = sorted(event_exclusion_summary["subject"].unique().tolist())

# Configure pandas to show full column content (no truncation)
pd.set_option('display.max_colwidth', None)

print(
    f"Detected {len(event_excluded_subjects)} subjects with annotation or marker anomalies."
)
if not event_exclusion_summary.empty:
    display(event_exclusion_summary)
else:
    print("All subjects pass event integrity validation.")

Detected 7 subjects with annotation or marker anomalies.


,subject,event_anomalies
0,38,Literature Flag: Known annotation/timing drift
1,88,R03: Task count mismatch (38/30) | R04: Task count mismatch (38/30) | R05: Task count mismatch (38/30) | R06: Task count mismatch (38/30) | R07: Task count mismatch (38/30) | R08: Task count mismatch (38/30) | R09: Task count mismatch (38/30) | R10: Task count mismatch (38/30) | R11: Task count mismatch (38/30) | R12: Task count mismatch (38/30) | R13: Task count mismatch (38/30) | R14: Task count mismatch (38/30)
2,89,R01: Baseline count mismatch (2) | R02: Baseline count mismatch (2) | R03: Task count mismatch (44/30); Literature Flag: Inconsistent labeling
3,92,R03: Task count mismatch (38/30); Literature Flag: Labeling errors | R04: Task count mismatch (38/30) | R05: Task count mismatch (38/30) | R06: Task count mismatch (38/30) | R07: Task count mismatch (38/30) | R08: Task count mismatch (38/30) | R09: Task count mismatch (38/30) | R10: Task count mismatch (38/30) | R11: Task count mismatch (38/30) | R12: Task count mismatch (38/30) | R13: Task count mismatch (38/30) | R14: Task count mismatch (38/30)
4,100,R03: Task count mismatch (24/30); Literature Flag: Inconsistent annotations | R04: Task count mismatch (24/30) | R05: Task count mismatch (24/30) | R06: Task count mismatch (24/30) | R07: Task count mismatch (24/30) | R08: Task count mismatch (24/30) | R09: Task count mismatch (24/30) | R10: Task count mismatch (24/30) | R11: Task count mismatch (24/30) | R12: Task count mismatch (24/30) | R13: Task count mismatch (24/30) | R14: Task count mismatch (24/30)
5,104,Literature Flag: Inconsistent annotations | R08: Task count mismatch (26/30)
6,106,Literature Flag: Inconsistent annotations | R05: Task count mismatch (9/30)


## Step 4: Clean Cohort Consolidation

In this final refinement step, we merge the subjects identified in **Step 2** (hardware mismatches) and **Step 3** (annotation/marker corruption) into a single, unified **exclusion list**. 

By subtracting this list from the original cohort, we obtain the **Clean Cohort**: a group of 102 subjects with perfect tensor uniformity (64 channels, 160 Hz) and reliable event synchronization, ready for robust machine learning training.

In [6]:
# Combine all excluded subjects from Steps 2 and 3
FINAL_EXCLUDED_SUBJECTS = sorted(list(set(hardware_excluded_subjects + event_excluded_subjects)))

# Generate the valid subject list
VALID_SUBJECTS = [s for s in SUBJECTS if s not in FINAL_EXCLUDED_SUBJECTS]

# Print final report
print("==========================================")
print("       FINAL COHORT REFINEMENT REPORT     ")
print("==========================================\n")
print(f"Total Subjects Audited   : {len(SUBJECTS)}")
print(f"Excluded Subjects (Total): {len(FINAL_EXCLUDED_SUBJECTS)}")
print(f"Final Clean Cohort Size  : {len(VALID_SUBJECTS)}")
print(f"\nFinal EXCLUDED_SUBJECTS List: {FINAL_EXCLUDED_SUBJECTS}")

# Validate against the 102 subject benchmark
if len(VALID_SUBJECTS) == 102:
    print("\n✅ VALIDATION SUCCESS: Cohort matches the 102-subject benchmark.")
else:
    print(f"\n⚠️ VALIDATION WARNING: Cohort size ({len(VALID_SUBJECTS)}) differs from expected (102).")

# Update the global registry (used in downstream notebooks)
%store VALID_SUBJECTS
%store FINAL_EXCLUDED_SUBJECTS

       FINAL COHORT REFINEMENT REPORT     

Total Subjects Audited   : 109
Excluded Subjects (Total): 7
Final Clean Cohort Size  : 102

Final EXCLUDED_SUBJECTS List: [38, 88, 89, 92, 100, 104, 106]

✅ VALIDATION SUCCESS: Cohort matches the 102-subject benchmark.
Stored 'VALID_SUBJECTS' (list)
Stored 'FINAL_EXCLUDED_SUBJECTS' (list)


## Data parsing after data validation 

In [7]:
print(f"Subjects to test: {VALID_SUBJECTS}")

Subjects to test: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 90, 91, 93, 94, 95, 96, 97, 98, 99, 101, 102, 103, 105, 107, 108, 109]


In [8]:

#! Define the runs you want to analyze.
# Ligne de base (Baseline), yeux ouverts
run_open_eyes = [1]
# Ligne de base (Baseline), yeux fermés
run_closed_eyes = [2]
# Motor execution: Open and close fist (Left vs. Right)
run_execution_hand = [3, 7, 11]
# Motor imagery: Imagine opening and closing fist (Left vs. Right)
run_imagery_hand = [4, 8, 12]
# Motor execution: Open and close both fists vs. both feet
run_execution_both_hands_feet = [5, 9, 13]
# Motor imagery: Imagine opening and closing both fists vs. both feet
run_imagery_both_hands_feet = [6, 10, 14]

# runs the entire set of runs for the BCI Competition IV dataset, which includes baseline, motor execution, and motor imagery tasks
runs = (
    run_open_eyes
    + run_closed_eyes
    + run_execution_hand
    + run_imagery_hand
    + run_execution_both_hands_feet
    + run_imagery_both_hands_feet
)

In [9]:
runs

[1, 2, 3, 7, 11, 4, 8, 12, 5, 9, 13, 6, 10, 14]

In [10]:
len(runs)

14

In [11]:
# ==============================================================================
# PHASE 2: TRIAL-BASED DATA PARSING & STRUCTURING
# ==============================================================================

# Initialize lists to store our trial data, labels, and provenance metadata
# X_list will hold the 3D data tensors (trials x channels x time)
X_list = []
# y_list will hold the numeric target labels for each trial
y_list = []
# metadata_records will track the original subject and run for every trial
metadata_records = []

# Define the time window for our trials (4 seconds at 160Hz = 641 samples)
tmin, tmax = 0.0, 4.0

# Flag to include/exclude rest (T0) periods in the final dataset
include_rest = False

print(f"Transforming EEG into 3D tensors for {len(VALID_SUBJECTS)} subjects...")

# Iterate through every subject in our clean cohort (102 subjects)
for subject in tqdm(VALID_SUBJECTS, desc="Processing Subjects"):

    # Define the specific Motor Imagery runs we want to classify:
    # 4, 8, 12: Imagine Left vs Right Fist (Unilateral)
    # 6, 10, 14: Imagine Both Fists vs Both Feet (Bilateral)
    imagery_runs = [4, 8, 12, 6, 10, 14]

    for run in imagery_runs:
        try:
            # 1. Load Raw Data: Verifies 160Hz, 64 channels, and applies montage
            # We use the existing 'load_eeg_data' helper from srcs/misc.py
            raw = load_eeg_data(subject, run, base_path=BASE_DATA_PATH)

            #! 2. Common Average Reference (CAR):
            # * We subtract the average of all channels from each channel.
            # * This reduces global noise and highlights local brain activity.
            #  !formula = (1/N) * sum(X_i) for i in 1 to N, where N is the number of channels and X_i is the signal from channel i.
            # ? internal noise reduction technique commonly used in EEG preprocessing to enhance signal quality.
            raw.set_eeg_reference("average", projection=False, verbose=False)

            # !3. Notch Filtering (60 Hz):
            # Targeted removal of power-line noise (hum) that exists at 60 Hz.
            # ? Humanas can be sensitive to 60Hz singnal interference created by electrical devides around them.
            #   by using the notch filter, we can remove this specific external signal interference from the EEG data
            #   improving the signal quality
            # * Firwin is a filter known in this domain for its ability to cut off specific frequencies while preserving the rest of the signal.
            raw.notch_filter(60.0, fir_design="firwin", verbose=False)

            # 4. Band-pass Filtering (8.0 - 30.0 Hz):
            # ?Keeps frequencies between 8 and 30 Hz (Alpha and Beta bands).
            # ?This is where the "Mu" rhythm associated with motor intent lives.
                # *alpha between 8 - 12 Hz, beta between 13 - 30 Hz
            # * skip_by_annotation="edge" ensures that we preserve the integrity of the signal at the edges of trials, avoiding artifacts from abrupt filtering.
                # ?an artifact is a signal that is not generated by the brain but rather by external sources
            raw.filter(
                8.0, 30.0, fir_design="firwin", skip_by_annotation="edge", verbose=False
            )

            # ?5. Dynamic Task Remapping:
            # PhysioNet uses T1/T2 generic markers. We map them to specific IDs:
            # *Separate the runs into two categories: unilateral (left/right fist) and bilateral (both fists/both feet).
            # !Run type 1 (4,8,12): T1 -> 0 (Left), T2 -> 1 (Right)
            if run in [4, 8, 12]:
                mapping = {"T1": 0, "T2": 1}
            # !Run type 2 (6,10,14): T1 -> 2 (Both Fists), T2 -> 3 (Both Feet)
            else:
                mapping = {"T1": 2, "T2": 3}

            # Optionally include 'Rest' as a 5th category (Class 4)
            if include_rest:
                mapping["T0"] = 4

            # Extract the actual event timings based on our mapping dictionary
            events, event_id = mne.events_from_annotations(
                raw, event_id=mapping, verbose=False
            )

            # 6. Trial Epoching:
            # !Cut the continuous signal into individual trials (0s to 4s relative to trigger).
            # baseline=None:             
                # ?already centered the signal around zero due to the band-pass filter, so we skip baseline subtraction. 
                # ?to now more about baseline correction: https://app.notion.com/p/jvalenci/total-perspective-vortex-3929d52658e080088fdcc42b60719b01?source=copy_link#3e59d52658e080fbb419d3a36f0f5e88 
            epochs = mne.Epochs(
                raw,
                events,
                event_id=event_id,
                tmin=tmin, # 00:00 seconds before the event
                tmax=tmax, # 04:00 seconds after the event
                # ?The baseline is the time window before the event that is used to calculate the baseline.
                baseline=None,
                preload=True,
                verbose=False,
            )

            # 7. Aggregate Data into NumPy arrays:
            # Convert MNE objects into raw arrays for scikit-learn.
            # X_run shape: (trials, 64 channels, 641 samples)
            X_run = epochs.get_data(copy=True)
            # y_run shape: (trials,) containing the mapped integers (0, 1, 2, or 3)
            y_run = epochs.events[:, -1]

            X_list.append(X_run)
            y_list.append(y_run)

            # 8. Metadata Collection:
            # Store trial provenance to ensure we can split data correctly (e.g. by subject).
            for i, label in enumerate(y_run):
                metadata_records.append(
                    {
                        "subject_id": subject,
                        "run_id": run,
                        # The trial index within this run (0 to n_trials-1)
                        "trial_id": i,
                        # The numeric label (0, 1, 2, or 3) e.g 0 -> T1, 1 -> T2, etc.
                        "label": label,
                        # The human-readable class name (e.g., "T1", "T2", etc.)
                        "class_name": list(mapping.keys())[
                            list(mapping.values()).index(label)
                        ],
                    }
                )

            # Memory Cleanup: EEG data is large; we delete objects after each run.
            del raw, epochs
            gc.collect()

        except Exception as e:
            # If a file is missing or corrupted, we print a warning and continue.
            print(f"Skipping Subject {subject} Run {run}: {e}")
            continue

# Final Data Consolidation:
# Stack all individual run arrays into a single large dataset.
if X_list:
    # Concatenate along the trial axis (axis 0)
    X = np.concatenate(X_list, axis=0)
    y = np.concatenate(y_list, axis=0)
    # Convert metadata into a DataFrame for easy querying and splitting
    df_metadata = pd.DataFrame(metadata_records)

    print("\n" + "=" * 42)
    print("      DATA STRUCTURING COMPLETE           ")
    print("=" * 42)
    print(f"Total Trials (N): {X.shape[0]}")
    print(f"Channels (C):    {X.shape[1]}")
    print(f"Samples (T):     {X.shape[2]}")
    print(f"Labels found:    {np.unique(y)}")
    print("=" * 42)
else:
    print("Error: No data was successfully processed.")

Transforming EEG into 3D tensors for 102 subjects...


Processing Subjects:   0%|          | 0/102 [00:00<?, ?it/s]

Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R04.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R08.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R12.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R06.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R10.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R14.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 mon

## Data Validation & Metadata Inspection

This section verifies that the generated 3D tensors ($X, y$) and the trial metadata are correctly structured for training.


In [12]:
df_metadata.columns

Index(['subject_id', 'run_id', 'trial_id', 'label', 'class_name'], dtype='str')

In [18]:
# Display the first few rows of our metadata to ensure trial provenance is correct
df_metadata['run_id'].unique()


array([ 4,  8, 12,  6, 10, 14])

In [27]:
df_metadata['class_name'][df_metadata['run_id'] == 4].unique()

<StringArray>
['T2', 'T1']
Length: 2, dtype: str

In [32]:
# Verify the class distribution across our dataset
print("Trial Count per Class:")
print(df_metadata['label'].value_counts().sort_index())

# Map back to human-readable names for clarity
class_map = {0: "Left Fist", 1: "Right Fist", 2: "Both Fists", 3: "Both Feet", 4: "Rest"}
print("\nHuman Readable Distribution:")
for label, count in df_metadata['label'].value_counts().sort_index().items():
    print(f" - {class_map.get(label, 'Unknown')}: {count} trials")


Trial Count per Class:
label
0    2304
1    2267
2    1726
3    1718
Name: count, dtype: int64

Human Readable Distribution:
 - Left Fist: 2304 trials
 - Right Fist: 2267 trials
 - Both Fists: 1726 trials
 - Both Feet: 1718 trials


In [ ]:
# ==============================================================================
# DEMONSTRATION: USING THE INTEGRATED LIBRARY FUNCTION
# ==============================================================================

# Import the new production-ready function from our library
from misc import load_and_parse_eeg

# Test with a small subset (e.g., first 2 valid subjects) to verify integration
test_subjects = VALID_SUBJECTS[:2]
test_runs = [4, 6] # One unilateral, one bilateral

print(f"Testing load_and_parse_eeg with subjects {test_subjects}...")

# Call the function with our specific parameters
X_test, y_test, df_meta_test = load_and_parse_eeg(
    subject_ids=test_subjects,
    run_ids=test_runs,
    base_path=BASE_DATA_PATH,
    include_rest=False
)

print("\nLibrary Function Test Results:")
# 4, 8, 12: Imagine Left vs Right Fist (Unilateral)
# 6, 10, 14: Imagine Both Fists vs Both Feet (Bilateral)
# !trials = (each run has 15 trials, we have 6 runs in total) for a total of 60 trials  
print(f"X shape: {X_test.shape} \n (Trials, Channels, Samples (641 / 640) = 4s)")
print(f"y shape: {y_test.shape} (Trials,)")
print(f"Unique Labels: {np.unique(y_test)}")
df_meta_test.head()


Testing load_and_parse_eeg with subjects [1, 2]...


Processing Subjects:   0%|          | 0/2 [00:00<?, ?it/s]

Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R04.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R06.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S002/S002R04.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S002/S002R06.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.

Library Function Test Results:
X shape: (60, 64, 641) 
 (Trials(0, 1, 2, 3), Channels, Samples (641 / 640) = 4s)
y shape: (60,) (Trials,)
Unique Labels: [0 1 2 3]


,subject_id,run_id,trial_id,label,class_name
0,1,4,0,1,T2
1,1,4,1,0,T1
2,1,4,2,0,T1
3,1,4,3,1,T2
4,1,4,4,1,T2
